In [1]:
!pip install "numpy==2.2.0" "scipy==1.13.0" --no-cache-dir --quiet
!pip install terratorch torchgeo --quiet
import os
os.kill(os.getpid(), 9)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 247.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.1/16.1 MB 221.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.2/38.2 MB 236.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
tsfresh 0.21.1 requires scipy>=1.14.0; python_version >= "3.10", but you have scipy 1.13.0 which is incompatible.
tenso

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import os
import sys
import json
import glob
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import terratorch
from terratorch.tasks import SemanticSegmentationTask
import albumentations as A
from albumentations.pytorch import ToTensorV2
import lightning.pytorch as pl
from lightning.pytorch.loggers import TensorBoardLogger
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping, LearningRateMonitor
import pandas as pd
from pathlib import Path
from scipy import ndimage

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
import os
import json
import lightning.pytorch as pl

BASE_PATH      = '/kaggle/input/datasets/akxran/dataset/data/'
PROCESSED      = BASE_PATH + 'processed/'
PROCESSED_PRED = BASE_PATH + 'processed_pred/'
OUT_DIR        = '/kaggle/working/'

os.makedirs(f'{OUT_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{OUT_DIR}/predictions', exist_ok=True)
os.makedirs(f'{OUT_DIR}/logs',        exist_ok=True)

EPOCHS         = 20
BATCH_SIZE     = 4
LR             = 6e-5
WEIGHT_DECAY   = 0.05
HEAD_DROPOUT   = 0.15
FREEZE_BACKBONE= False
BACKBONE       = "prithvi_eo_v2_300_tl"
NUM_FRAMES     = 1
NUM_CLASSES    = 2
BAND_INDICES   = [0, 2, 3, 4, 5, 6]
BANDS          = [0, 1, 2, 3, 4, 5]
CLASS_WEIGHTS  = [0.15, 0.85]
FLOOD_THRESHOLD= 0.30
TTA_ENABLED    = True
POST_PROCESS   = True
SEED           = 42

with open(BASE_PATH + 'norm_stats.json') as f:
    norm_stats = json.load(f)
ALL_MEANS = norm_stats['mean']
ALL_STDS  = norm_stats['std']
MEANS = [ALL_MEANS[i] for i in BAND_INDICES]
STDS  = [ALL_STDS[i]  for i in BAND_INDICES]

pl.seed_everything(SEED)
print("Working Hopefully")
print(f"Band indices:  {BAND_INDICES}")
print(f"Means: {[round(m,2) for m in MEANS]}")
print(f"Stds:  {[round(s,2) for s in STDS ]}")

In [ ]:
import glob
import rasterio
import pandas as pd

tif_files = sorted(glob.glob('/kaggle/input/datasets/akxran/dataset/data/raw/**/*.tif', recursive=True))
print(f"Total GeoTIFF files found: {len(tif_files)}")

records = []
for fpath in tif_files[:5]:
    with rasterio.open(fpath) as src:
        records.append({
            'file'      : os.path.basename(fpath),
            'bands'     : src.count,
            'width'     : src.width,
            'height'    : src.height,
            'crs'       : str(src.crs),
            'dtype'     : str(src.dtypes[0]),
            'resolution': round(src.res[0], 4),
        })

df_inv = pd.DataFrame(records)
print(df_inv.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import rasterio

BAND_NAMES = ['Blue', 'Green', 'Red','NIR', 'SWIR1', 'SWIR2']
colors     = ['#1f77b4','#2ca02c','#d62728','#8B0000','#9467bd','#8c564b']

with rasterio.open(tif_files[0]) as src:
    all_bands = src.read().astype(np.float32)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Per-Band Pixel Value Distributions (Raw)', fontsize=14, fontweight='bold')
axes = axes.flatten()

for i, (name, color) in enumerate(zip(BAND_NAMES, colors)):
    vals = all_bands[i][all_bands[i] > 0].flatten()
    axes[i].hist(vals, bins=80, color=color, alpha=0.75, edgecolor='none')
    axes[i].set_title(name, fontweight='bold')
    axes[i].set_xlabel('Pixel Value')
    axes[i].set_ylabel('Count')
    axes[i].ticklabel_format(style='sci', axis='y', scilimits=(0, 0))

plt.tight_layout()
plt.savefig('/kaggle/working/band_histograms.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import os
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import lightning.pytorch as pl

def flood_collate_fn(batch):
    images = torch.stack([b['image'] for b in batch])
    masks  = torch.stack([b['mask']  for b in batch])
    return {"image": images, "mask": masks}

def pred_collate_fn(batch):
    images    = torch.stack([b['image']    for b in batch])
    filenames = [b['filename'] for b in batch]
    return {"image": images, "filename": filenames}

class FloodNpzDataset(Dataset):
    def __init__(self, file_list, transform=None, is_predict=False):
        self.files      = file_list
        self.transform  = transform
        self.is_predict = is_predict
        self.means      = np.array(MEANS, dtype=np.float32).reshape(-1, 1, 1)
        self.stds       = np.array(STDS,  dtype=np.float32).reshape(-1, 1, 1)

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        data  = np.load(self.files[idx])
        image = data['image'].astype(np.float32)
        image = image[BAND_INDICES, :, :]
        image = (image - self.means) / (self.stds + 1e-6)

        if self.is_predict:
            return {
                "image":    torch.from_numpy(image),
                "filename": os.path.basename(self.files[idx])
            }

        label = data['label'].astype(np.int64)

        if self.transform:
            image_hwc = image.transpose(1, 2, 0)
            augmented = self.transform(image=image_hwc, mask=label)
            image = augmented['image'].float()
            label = augmented['mask'].long()
        else:
            image = torch.from_numpy(image)
            label = torch.from_numpy(label)

        return {"image": image, "mask": label}

train_transforms = A.Compose([
    A.D4(p=1.0),
    A.Affine(
        scale=(0.9, 1.1),
        translate_percent=(-0.05, 0.05),
        rotate=(-15, 15),
        shear=(-5, 5),
        p=0.4,
        border_mode=0,
    ),
    A.ElasticTransform(alpha=50, sigma=10, p=0.2),
    A.GridDistortion(num_steps=5, distort_limit=0.2, p=0.2),
    A.GaussNoise(std_range=(0.01, 0.03), p=0.3),
    A.GaussianBlur(blur_limit=(3, 5), p=0.15),
    A.RandomBrightnessContrast(
        brightness_limit=0.15,
        contrast_limit=0.15,
        p=0.3,
    ),
    A.CoarseDropout(
        num_holes_range=(4, 8),
        hole_height_range=(16, 48),
        hole_width_range=(16, 48),
        fill=0,
        p=0.3,
    ),
    ToTensorV2(),
])

val_transforms = A.Compose([ToTensorV2()])

all_files = sorted([
    PROCESSED + f for f in os.listdir(PROCESSED) if f.endswith('.npz')
])
pred_files = sorted([
    PROCESSED_PRED + f for f in os.listdir(PROCESSED_PRED) if f.endswith('.npz')
])

n_total = len(all_files)
n_train = int(0.8 * n_total)
n_val   = n_total - n_train

train_files = all_files[:n_train]
val_files   = all_files[n_train:]

print(f"Total files    : {n_total}")
print(f"Train samples  : {len(train_files)}")
print(f"Val samples    : {len(val_files)}")
print(f"Predict samples: {len(pred_files)}")

train_dataset = FloodNpzDataset(train_files, transform=train_transforms)
val_dataset   = FloodNpzDataset(val_files,   transform=val_transforms)
pred_dataset  = FloodNpzDataset(pred_files,  transform=None, is_predict=True)

class FloodDataModule(pl.LightningDataModule):
    def __init__(self, batch_size=4, num_workers=2):
        super().__init__()
        self.batch_size  = batch_size
        self.num_workers = num_workers

    def train_dataloader(self):
        return DataLoader(train_dataset, batch_size=self.batch_size,
                          shuffle=True,  num_workers=self.num_workers,
                          pin_memory=True, collate_fn=flood_collate_fn)

    def val_dataloader(self):
        return DataLoader(val_dataset,   batch_size=self.batch_size,
                          shuffle=False, num_workers=self.num_workers,
                          pin_memory=True, collate_fn=flood_collate_fn)

    def test_dataloader(self):
        return DataLoader(val_dataset,   batch_size=self.batch_size,
                          shuffle=False, num_workers=self.num_workers,
                          pin_memory=True, collate_fn=flood_collate_fn)

    def predict_dataloader(self):
        return DataLoader(pred_dataset,  batch_size=1,
                          shuffle=False, num_workers=self.num_workers,
                          collate_fn=pred_collate_fn)

datamodule = FloodDataModule(batch_size=BATCH_SIZE, num_workers=2)

sample = train_dataset[0]
print(f"\nSample image : {sample['image'].shape}")
print(f"Sample mask  : {sample['mask'].shape}")
print(f"Label values : {torch.unique(sample['mask']).tolist()}")
print("\nITS RUNNING.....WE ARE FLYING....")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.patches as mpatches

flood_pixels = 0
total_pixels = 0

for fpath in train_files[:30]:          
    data  = np.load(fpath)
    label = data['label']
    flood_pixels += int(label.sum())
    total_pixels += label.size

bg_pixels   = total_pixels - flood_pixels
percentages = [bg_pixels / total_pixels * 100, flood_pixels / total_pixels * 100]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Class Distribution — Flood vs Background', fontsize=14, fontweight='bold')

bars = axes[0].bar(['Background', 'Flood'], [bg_pixels, flood_pixels],
                   color=['#2196F3', '#F44336'], edgecolor='white', width=0.5)
for bar, pct in zip(bars, percentages):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 0.5,
                 f'{pct:.1f}%', ha='center', va='center',
                 color='white', fontweight='bold', fontsize=13)
axes[0].set_ylabel('Pixel Count')
axes[0].set_title('Pixel Count by Class')
axes[0].ticklabel_format(style='sci', axis='y', scilimits=(0, 0))

axes[1].pie([bg_pixels, flood_pixels], labels=['Background', 'Flood'],
            autopct='%1.1f%%', colors=['#2196F3', '#F44336'],
            startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Class Proportion')

plt.tight_layout()
plt.savefig('/kaggle/working/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Flood pixels : {flood_pixels:,}  ({percentages[1]:.2f}%)")
print(f"Background   : {bg_pixels:,}  ({percentages[0]:.2f}%)")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

BAND_INDICES = [0, 2, 3, 4, 5, 6]
n_show = 6
fig, axes = plt.subplots(2, n_show, figsize=(n_show * 3, 6))
fig.suptitle('Sample Tiles  —  Normalised Image (top)  |  Flood Label (bottom)',
             fontsize=13, fontweight='bold')

for i in range(n_show):
    sample = train_dataset[i]
    img    = sample['image'].numpy()
    lbl    = sample['mask'].numpy()

    def norm01(x):
        return np.clip((x - x.min()) / (x.max() - x.min() + 1e-6), 0, 1)

    rgb_tile = np.stack([norm01(img[2]), norm01(img[1]), norm01(img[0])], axis=-1)

    axes[0, i].imshow(rgb_tile)
    axes[0, i].set_title(f'Tile {i}', fontsize=9)
    axes[0, i].axis('off')

    axes[1, i].imshow(lbl, cmap='Reds', vmin=0, vmax=1)
    axes[1, i].axis('off')

plt.tight_layout()
plt.savefig('/kaggle/working/tile_gallery.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

flood_ratios = []
for fpath in train_files:
    data = np.load(fpath)
    flood_ratios.append(float(data['label'].mean()))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Dataset Overview Before Training', fontsize=14, fontweight='bold')

# Flood ratio histogram
axes[0].hist(flood_ratios, bins=40, color='#F44336', alpha=0.8, edgecolor='white')
axes[0].axvline(np.mean(flood_ratios), color='black', linestyle='--', linewidth=2,
                label=f'Mean = {np.mean(flood_ratios):.3f}')
axes[0].set_xlabel('Flood Pixel Fraction per Tile')
axes[0].set_ylabel('Number of Tiles')
axes[0].set_title('Per-Tile Flood Ratio Distribution')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Split summary bar
splits = ['Train', 'Validation', 'Predict']
counts = [len(train_files), len(val_files), len(pred_files)]
colors = ['#1A56A4', '#2E86C1', '#85C1E9']
bars   = axes[1].bar(splits, counts, color=colors, edgecolor='white', width=0.5)
for bar, cnt in zip(bars, counts):
    axes[1].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 0.3, str(cnt),
                 ha='center', va='bottom', fontweight='bold')
axes[1].set_ylabel('Number of Tiles')
axes[1].set_title('Dataset Split Summary')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/dataset_overview.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Train tiles  : {len(train_files)}")
print(f"Val tiles    : {len(val_files)}")
print(f"Predict tiles: {len(pred_files)}")
print(f"Mean flood ratio (train): {np.mean(flood_ratios):.4f}")

In [ ]:
import torch
import torch.nn as nn
import gc
import numpy as np
from scipy import ndimage

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
gc.collect()
torch.cuda.empty_cache()

print(f"GPU free  : {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")
print(f"GPU total : {torch.cuda.mem_get_info()[1]/1e9:.1f} GB")

pl.seed_everything(SEED)

logger = TensorBoardLogger(save_dir=OUT_DIR, name="flood_final")

checkpoint_callback = ModelCheckpoint(
    monitor="val/mIoU",
    mode="max",
    dirpath=os.path.join(OUT_DIR, "flood_final", "checkpoints"),
    filename="best-{epoch:02d}-{val_mIoU:.4f}",
    save_top_k=1,
    verbose=True,
)
early_stop_callback = EarlyStopping(
    monitor="val/mIoU",
    mode="max",
    patience=10,
    verbose=True,
)
lr_monitor = LearningRateMonitor(logging_interval="epoch")

# TRAINER
trainer = pl.Trainer(
    accelerator="auto",
    strategy="auto",
    devices="auto",
    precision="bf16-mixed",
    num_nodes=1,
    logger=logger,
    max_epochs=EPOCHS,
    check_val_every_n_epoch=1,
    log_every_n_steps=5,
    enable_checkpointing=True,
    callbacks=[checkpoint_callback, early_stop_callback, lr_monitor],
    gradient_clip_val=1.0,
)

# MODEL
decoder_args = dict(
    decoder="UperNetDecoder",
    decoder_channels=256,
    decoder_scale_modules=True,
)
necks = [dict(name="ReshapeTokensToImage", effective_time_dim=NUM_FRAMES)]
backbone_args = dict(
    backbone_pretrained=True,
    backbone=BACKBONE,
    backbone_bands=BANDS,
    backbone_num_frames=NUM_FRAMES,
)
model_args = dict(
    **backbone_args,
    **decoder_args,
    num_classes=NUM_CLASSES,
    head_dropout=HEAD_DROPOUT,
    necks=necks,
    rescale=True,
)

model = SemanticSegmentationTask(
    model_args=model_args,
    plot_on_val=False,
    class_weights=CLASS_WEIGHTS,
    loss="dice",
    lr=LR,
    optimizer="AdamW",
    optimizer_hparams=dict(weight_decay=WEIGHT_DECAY),
    ignore_index=-1,
    freeze_backbone=FREEZE_BACKBONE,
    freeze_decoder=False,
    model_factory="EncoderDecoderFactory",
    scheduler="CosineAnnealingLR",
    scheduler_hparams=dict(T_max=EPOCHS, eta_min=1e-7),
)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params:     {total_params/1e6:.1f}M")
print(f"Trainable params: {trainable_params/1e6:.1f}M")
print("Back with a Bang")

# TRAINING
print("\n" + "=" * 60)
print(f"Starting training: {EPOCHS} epochs, batch={BATCH_SIZE}")
print(f"Backbone: {BACKBONE}")
print(f"Loss: dice | Weights: {CLASS_WEIGHTS}")
print("=" * 60)

trainer.fit(model, datamodule=datamodule)

best_ckpt = checkpoint_callback.best_model_path
best_miou = checkpoint_callback.best_model_score
print(f"\nTraining complete!")
print(f"Best checkpoint: {best_ckpt}")
print(f"Best val mIoU:   {best_miou:.4f}")

# TEST EVALUATION
print("\n" + "=" * 60)
print("Bang Incoming:")
test_results = trainer.test(model, datamodule=datamodule, ckpt_path=best_ckpt)
print("\n Bang Bang")
for k, v in test_results[0].items():
    print(f"  {k}: {v:.4f}")

# THRESHOLD OPTIMIZATION
print("\n" + "=" * 60)

model.eval()
model = model.cuda()

all_probs  = []
all_labels = []

with torch.no_grad():
    for batch in datamodule.val_dataloader():
        images = batch['image'].cuda()
        masks  = batch['mask']
        output = model(images)
        probs  = torch.softmax(output.output, dim=1)[:, 1]
        all_probs.append(probs.cpu())
        all_labels.append(masks)

all_probs  = torch.cat(all_probs,  dim=0).numpy()
all_labels = torch.cat(all_labels, dim=0).numpy()

best_thresh = 0.5
best_iou    = 0.0

for thresh in np.arange(0.25, 0.75, 0.05):
    preds        = (all_probs > thresh).astype(np.int64)
    intersection = ((preds == 1) & (all_labels == 1)).sum()
    union        = ((preds == 1) | (all_labels == 1)).sum()
    iou          = intersection / (union + 1e-6)
    if iou > best_iou:
        best_iou    = iou
        best_thresh = thresh
    print(f"  threshold={thresh:.2f}  IoU_1={iou:.4f}")

print(f"\nBest threshold : {best_thresh:.2f}")
print(f"Best IoU_1     : {best_iou:.4f}")
FLOOD_THRESHOLD = best_thresh

# 8-WAY TTA INFERENCE
def tta_predict(model, image):
    preds = []
    with torch.no_grad():
        for k in range(4):
            img_rot = torch.rot90(image, k, dims=[-2, -1])
            for flip in [False, True]:
                img_aug = img_rot.flip(-1) if flip else img_rot
                out     = model(img_aug.cuda())
                prob    = torch.softmax(out.output, dim=1)[:, 1]
                if flip:
                    prob = prob.flip(-1)
                prob = torch.rot90(
                    prob.unsqueeze(1), -k, dims=[-2, -1]
                ).squeeze(1)
                preds.append(prob.cpu())
    return torch.stack(preds, dim=0).mean(dim=0)

def post_process(mask, min_size=50):
    labeled, n = ndimage.label(mask)
    for i in range(1, n + 1):
        if (labeled == i).sum() < min_size:
            mask[labeled == i] = 0
    return ndimage.binary_fill_holes(mask).astype(np.uint8)

#8-Way TTA
print("\n" + "=" * 60)
os.makedirs(f'{OUT_DIR}/predictions', exist_ok=True)
prediction_results = {}

for batch in datamodule.predict_dataloader():
    images    = batch['image']
    filenames = batch['filename']
    avg_prob  = tta_predict(model, images)

    for i, fname in enumerate(filenames):
        prob_map  = avg_prob[i].numpy()
        pred_mask = (prob_map > FLOOD_THRESHOLD).astype(np.uint8)
        pred_mask = post_process(pred_mask, min_size=50)
        prediction_results[fname] = pred_mask
        print(f"  {fname}: flood_ratio={pred_mask.mean():.3f}")

# SAVE PREDICTIONS
for fname, mask in prediction_results.items():
    save_name = fname.replace('.npz', '_pred.npy')
    np.save(f'{OUT_DIR}/predictions/{save_name}', mask)

print(f"\nSaved {len(prediction_results)} predictions")
print(f"Location: {OUT_DIR}/predictions/")
print("\n" + "=" * 60)
print("FINAL SUMMARY")
print("=" * 60)
print(f"Best val mIoU   : {best_miou:.4f}")
print(f"Best checkpoint : {best_ckpt}")
print(f"Flood threshold : {FLOOD_THRESHOLD:.2f}")
print(f"TTA             : 8-way")
print(f"Post-processing : enabled (min_size=50)")
print("=" * 60)

In [ ]:
import pandas as pd
import numpy as np
import os

predictions_dir = f'{OUT_DIR}/predictions/'
prediction_results = {}

for fname in os.listdir(predictions_dir):
    if fname.endswith('_pred.npy'):
        mask = np.load(os.path.join(predictions_dir, fname))
        original_name = fname.replace('_pred.npy', '.npz')
        prediction_results[original_name] = mask

print(f"Loaded: {len(prediction_results)} predictions")

def mask_to_rle(mask):
    pixels = mask.flatten(order='F')
    pixels = np.concatenate([[0], pixels, [0]])
    runs   = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    if len(runs) == 0:
        return ''
    return ' '.join(str(r) for r in runs)

def rle_to_mask(rle_string, shape=(512, 512)):
    if not rle_string or rle_string.strip() == '':
        return np.zeros(shape, dtype=np.uint8)
    nums = list(map(int, rle_string.split()))
    mask = np.zeros(shape[0] * shape[1], dtype=np.uint8)
    for i in range(0, len(nums), 2):
        start  = nums[i] - 1
        length = nums[i + 1]
        mask[start:start + length] = 1
    return mask.reshape(shape, order='F')

rows = []
for fname, mask in prediction_results.items():
    image_id = fname.replace('.npz', '')
    rle      = mask_to_rle(mask)
    rows.append({'id': image_id, 'rle_mask': rle})
    print(f"  {image_id}: flood_pixels={mask.sum()}")

submission = pd.DataFrame(rows, columns=['id', 'rle_mask'])
submission = submission.sort_values('id').reset_index(drop=True)

submission_path = f'{OUT_DIR}/submission.csv'
submission.to_csv(submission_path, index=False)

print(f"\nTotal rows: {len(submission)}")
print("\nSample:")
print(submission[['id', 'rle_mask']].head(3).to_string())

all_correct = True
for _, row in submission.iterrows():
    fname    = row['id'] + '.npz'
    decoded  = rle_to_mask(row['rle_mask'])
    original = prediction_results[fname]
    match    = (decoded == original).all()
    if not match:
        print(f" {row['id']} — RLE mismatch!")
        all_correct = False

if all_correct:
    print(f"  Al{len(submission)} masks verified correctly")

assert len(submission) == 19, f"Expected 19 rows, got {len(submission)}"
assert list(submission.columns) == ['id', 'rle_mask'], "Wrong columns"
print(f"\n Row count    : {len(submission)}")
print(f" Columns      : {list(submission.columns)}")
print(f" File saved   : {submission_path}")
print("\nOh The lord has come!!!")